In [71]:
import gradio as gr
from __future__ import annotations
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from PIL import Image

In [72]:
# ----------------------------
# Object, first time run
# ----------------------------

@dataclass(frozen=True)
class Obj:
    name: str
    color: str
    shape: str  # "cube", "triangle", "block" (generic)
    size: str  # "small", "medium", "large"


In [ ]:
# ----------------------------
# World, first time run
# ----------------------------

@dataclass
class World:
    objects: Dict[str, Obj]
    objects_inside_box: List[str] # list of object names that are inside the box
    whoIsUnder: Dict[str, Optional[str]] # on[x] = y means x is on y; None means on table
    #If you have on["block_a"] = "block_b", it means block_a is physically sitting on top of block_b
    rightNeighbors: Dict[str, List[str]] #horizontalNeighbors[x] = [y1, y2, ...] means x is immediately to the left of y1, y2, ...
    # empty list means its picked up, non emeans its the most right object
    holding: Optional[str] = None

    def is_clear(self, obj_name: str) -> bool:
        """True if nothing is on top of obj_name."""
        # return all(support != obj_name for support in self.on.values())
        for support in self.whoIsUnder.values():
            if support == obj_name:
                return False
        return True

    def top_of(self, obj_name: str) -> Optional[str]:
        """Return object that is on top of obj_name (if any)."""
        for x, support in self.whoIsUnder.items():
            print(x,support)
            if support == obj_name:
                return x
        return None

    def describe(self) -> str:
        #describes all relationships between objects and if world agent is holding something
        lines = []
        for x in sorted(self.objects):
            support = self.whoIsUnder.get(x, None)
            if support is None:
                lines.append(f"{x} is on the table")
            else:
                lines.append(f"{x} is on {support}")
        lines.append(f"Holding: {self.holding}")
        return "\n".join(lines)

    ##########
    
    def pickup(self, x: str) -> None:
        # 2. Capture X's current right neighbors before we clear them
        rights_of_x = self.rightNeighbors.get(x, [])
        
        # 3. Update the left neighbor to point to X's right neighbors
        for name, neighbors in self.rightNeighbors.items():
            if neighbors and x in neighbors:
                # Remove x from the left neighbor's list
                neighbors.remove(x)
                # Add X's old right neighbors to this left neighbor
                for r in rights_of_x:
                    if r not in neighbors:
                        neighbors.append(r)
        
        # 4. Update state: X is now being held
        self.holding = x
        self.whoIsUnder[x] = None
        
        # 5. Clear X's own horizontal links (it's in the air)
        self.rightNeighbors[x] = []
        

    def put_on(self, x: str, y: str) -> None:
        #put down obj on another obj, if holding something and if target object isnt under something else / blocked
        if self.holding != x:
            raise RuntimeError(f"Not holding {x}.")
        if not self.is_clear(y):
            raise RuntimeError(f"Cannot place on {y}: {y} not clear.")
        self.whoIsUnder[x] = y
        self.holding = None
    
    def put_down(self, x: str) -> None:
        #put down obj on table, if holding something and if target object isnt under something else / blocked
        if self.holding != x:
            raise RuntimeError(f"Not holding {x}.")
        
        # 1. Identify objects currently on the table (excluding the one in hand)
        on_table = [name for name, support in self.whoIsUnder.items() 
                    if support is None and name != self.holding]
        
        # 2. Find the rightmost object (where neighbors list is [])
        rightmost_obj = None
        for obj_name in on_table:
            neighbors = self.rightNeighbors.get(obj_name)
            if not self.rightNeighbors.get(obj_name, []):
                rightmost_obj = obj_name
                break
        
        # 3. Update the horizontal chain
        if rightmost_obj:
            self.rightNeighbors[rightmost_obj] = [x]
        
        # 4. x is now the new end of the line
        self.rightNeighbors[x] = []

        # 5. Final state update
        self.whoIsUnder[x] = None
        self.holding = None


# Initialize world state
def create_initial_world():
    return World(
        objects={
            "c1": Obj("c1", "red", "cube", "medium"),
            "b2": Obj("b2", "blue", "cube", "medium"),
            "p1": Obj("p1", "blue", "triangle", "large"),
            "c2": Obj("c2", "green", "triangle", "small"),
            "l1": Obj("l1", "cyan", "circle", "medium"),
            "r1": Obj("r1", "magenta", "rectangle", "large"),
            "s1": Obj("s1", "yellow", "pentagon", "medium"),
            "s2": Obj("s2", "red", "pentagon", "large"),
            "l2": Obj("l2", "blue", "circle", "small"),
            "l3": Obj("l3", "green", "circle", "large"),
            "b3": Obj("b3", "cyan", "cube", "small"),
            "c3": Obj("c3", "magenta", "triangle", "medium"),
            "c4": Obj("c4", "yellow", "cube", "small"),
        },
        whoIsUnder={
            "p1": "c1",
            "c1": None,
            "b2": None,
            "c2": None,
            "l1": "c2",
            "r1": None,
            "s1": None,
            "s2": None,
            "l2": None,
            "l3": "l2",
            "b3": "l3",
            "c3": None,
            "c4": None
        },
        rightNeighbors={
            "p1": ["b2"],
            "c1": ["b2"],
            "b2": ["c2","l1"],
            "c2": ["r1"],
            "l1": ["r1"],
            "r1": ["s1"],
            "s1": ["s2"],
            "s2": ["l2","l3","b3"],
            "l2": ["c3"],
            "l3": ["c3"],
            "b3": ["c3"],
            "c3": ["c4"],
            "c4": []
        },
        objects_inside_box=["c1", "p1"]
    )

# Global world state
world_state = create_initial_world()

In [74]:
# ----------------------------------------
# STEP 3) Planning
# ----------------------------------------

def plan_pickup(world: World, x: str) -> List[Tuple[str, str, Optional[str]]]:
    """Plan to pick up x. If x is not clear, move blockers to table first."""
    plan = []
    blocker = world.top_of(x)
    if blocker is not None:
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))
    plan.append(("pickup", x, None))
    return plan


def plan_put(world: World, x: str, y: str) -> List[Tuple[str, str, Optional[str]]]:
    """Plan to put x on y, ensuring both are feasible."""
    plan = []
    blocker = world.top_of(y)
    if blocker is not None:
        plan += plan_pickup(world, blocker)
        plan.append(("put_on", blocker, None))
    plan += plan_pickup(world, x)
    plan.append(("put_on", x, y))
    return plan


def execute_plan(world: World, plan: List[Tuple[str, str, Optional[str]]]) -> None:
    for action, obj, target in plan:
        if action == "pickup":
            world.pickup(obj)
        elif action == "put_on":
            if target is None:
                if world.holding != obj:
                    raise RuntimeError("Planner/executor mismatch.")
                world.whoIsUnder[obj] = None
                world.holding = None
            else:
                world.put_on(obj, target)
        elif action == "put_down":
            if world.holding != obj:
                raise RuntimeError("Planner/executor mismatch.")
            world.whoIsUnder[obj] = None
            world.holding = None
        else:
            raise ValueError(f"Unknown action: {action}")

In [75]:
# ----------------------------------------
# STEP 2) Parsing
# ----------------------------------------

COLORS = {"red", "green", "blue", "yellow", "magenta", "cyan"}
SHAPES = {"triangle", "cube", "rectangle", "circle", "pentagon", "block"}
SIZES = {"small", "medium", "large"}

def parse_command(text: str) -> dict:
    """Minimal rule-based parser for demo commands."""
    t = text.lower().replace("?", "").strip()
    tokens = t.split()

    if t.startswith("pick up"):
        color = next((w for w in tokens if w in COLORS), None)
        shape = next((w for w in tokens if w in SHAPES), "block")
        size = next((w for w in tokens if w in SIZES), "medium")
        return {"intent": "PICKUP", "ref": {"color": color, "shape": shape, "size": size}}

    if t.startswith("put"):
        if "on" in tokens:
            on_i = tokens.index("on")
            left = tokens[1:on_i]
            right = tokens[on_i+1:]

            color_x = next((w for w in left if w in COLORS), None)
            shape_x = next((w for w in left if w in SHAPES), "block")
            size_x = next((w for w in left if w in SIZES), "medium")
            color_y = next((w for w in right if w in COLORS), None)
            shape_y = next((w for w in right if w in SHAPES), "block")
            size_y = next((w for w in right if w in SIZES), "medium")

            return {
            "intent": "PUT_ON",
            "x": {"color": color_x, "shape": shape_x, "size": size_x},
            "y": {"color": color_y, "shape": shape_y, "size": size_y},
        }
        elif "down" in tokens:
            #put down
            on_i = tokens.index("down")
            left = tokens[1:on_i]
            right = tokens[on_i+1:]

            color_x = next((w for w in left if w in COLORS), None)
            shape_x = next((w for w in left if w in SHAPES), "block")
            size_x = next((w for w in left if w in SIZES), "medium")
            color_y = next((w for w in right if w in COLORS), None)
            shape_y = next((w for w in right if w in SHAPES), "block")
            size_y = next((w for w in right if w in SIZES), "medium")
            
            return {
            "intent": "PUT_DOWN",
            "x": {"color": color_x, "shape": shape_x, "size": size_x},
            "y": {"color": color_y, "shape": shape_y, "size": size_y},
        }

    raise ValueError("Unknown command format.")

#finds object names matching the requested properties, and returns a list of matches
def resolve_ref(world: World, color: Optional[str], shape: Optional[str]) -> List[str]:
    """Return object names matching the requested properties."""
    matches = []
    for name, obj in world.objects.items():
        if color is not None and obj.color != color:
            continue
        if shape is not None:
            if shape != "block" and obj.shape != shape:
                continue
        matches.append(name)
    return matches

#returns single, no or multiple matches for the given description of an object
def choose_unique(matches: List[str], what: str) -> str:
    if not matches:
        raise ValueError(f"I can't find any {what} that matches your description.")
    if len(matches) > 1:
        raise ValueError(f"I don't understand which {what} you mean: {matches}")
    return matches[0]



#this calls everything
def interpret_and_act(world: World, utterance: str) -> str:
    """Execute command and return response message."""
    parsed = parse_command(utterance)

    if parsed["intent"] == "PICKUP":
        ref = parsed["ref"]
        matches = resolve_ref(world, ref["color"], ref["shape"])
        x = choose_unique(matches, f"{ref['color'] or ''} {ref['shape']}".strip())
        plan = plan_pickup(world, x)
        execute_plan(world, plan)
        return f"OK. Picked up {x}. Plan: {plan}"

    elif parsed["intent"] == "PUT_ON":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"])
        my = resolve_ref(world, parsed["y"]["color"], parsed["y"]["shape"])
        x = choose_unique(mx, f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        y = choose_unique(my, f"{parsed['y']['color'] or ''} {parsed['y']['shape']}".strip())
        plan = plan_put(world, x, y)
        execute_plan(world, plan)
        return f"OK. Put {x} on {y}. Plan: {plan}"
    
    elif parsed["intent"] == "PUT_DOWN":
        mx = resolve_ref(world, parsed["x"]["color"], parsed["x"]["shape"])
        x = choose_unique(mx, f"{parsed['x']['color'] or ''} {parsed['x']['shape']}".strip())
        plan = [("put_down", x, None)]
        execute_plan(world, plan)
        return f"OK. Put down {x}. Plan: {plan}"

    else:
        raise ValueError("Unsupported intent.")


In [76]:
# ----------------------------------------
# STEP 4) Visualization
# ----------------------------------------

size_map = {
    "small": 0.3,
    "medium": 0.5,
    "large": 0.8
}

#helper
def get_width(obj):
    s = size_map[obj.size]
    
    if obj.shape == "rectangle":
        return s
    elif obj.shape == "circle":
        return s
    elif obj.shape == "triangle":
        return s
    elif obj.shape == "pentagon":
        return s
    else:  # cube/block
        return s

def get_height(obj):
    s = size_map[obj.size]
    
    if obj.shape == "triangle":
        return s * 1.2
    elif obj.shape == "cube":
        return s
    elif obj.shape == "rectangle":
        return s / 1.5
    elif obj.shape == "circle":
        return s
    elif obj.shape == "pentagon":
        return s * 0.6
    else:  # block
        return s

def draw_object(obj, center_x, y_pos):
    w = get_width(obj)
    h = get_height(obj)
    color = obj.color
    left = center_x - w / 2
    if obj.shape == "triangle":
        points = [[left, y_pos], [left + w, y_pos], [center_x, y_pos + h]]
        patch = patches.Polygon(points, closed=True, edgecolor='black', facecolor=color, linewidth=2)
    elif obj.shape == "circle":
        r = w / 2
        patch = patches.Circle((center_x, y_pos + r), r, edgecolor='black', facecolor=color, linewidth=2)
    elif obj.shape == "pentagon":
        points = [
            [center_x, y_pos + h],           # top
            [left + w * 0.9, y_pos + h * 0.7], # right upper
            [left + w * 0.7, y_pos],         # right lower
            [left + w * 0.3, y_pos],         # left lower
            [left + w * 0.1, y_pos + h * 0.7], # left upper
        ]
        patch = patches.Polygon(points, closed=True, edgecolor='black', facecolor=color, linewidth=2)
    else: # cube, rectangle, block
        patch = patches.Rectangle((left, y_pos), w, h, edgecolor='black', facecolor=color, linewidth=2)
    return patch
    


def visualize_world(world) -> Image.Image:
    """Create a visual representation with dynamic spacing and limits."""
    
    # 1. ORGANIZE STACKS
    stacks = {}
    on_table = [name for name, support in world.whoIsUnder.items() 
                if support is None and name != world.holding]
    
    # Identify which objects are actually "on the left" 
    # (i.e., they are on the table and no one has them as a rightNeighbor)
    all_rights = set()
    for neighbors in world.rightNeighbors.values():
        all_rights.update(neighbors)

    leftmost_objs = [obj for obj in on_table if obj not in all_rights]

    # We will build an ordered list of table objects by traversing rightNeighbors
    ordered_on_table = []
    visited = set()
    
    def traverse(current):
        if current in visited or current not in on_table:
            return
        visited.add(current)
        ordered_on_table.append(current)
        # Look at neighbors; filter to only those on the table
        for neighbor in world.rightNeighbors.get(current, []):
            traverse(neighbor)

    for start_node in sorted(leftmost_objs): # Sort starts alphabetically just in case of ties
        traverse(start_node)

    for table_obj in ordered_on_table:
        stack = [table_obj]
        current = table_obj
        while True:
            top = world.top_of(current)
            if top is None: break
            stack.append(top)
            current = top
        stacks[table_obj] = stack

    # 2. PRE-CALCULATE DYNAMIC X-POSITIONS
    stack_positions = {}
    current_x = 0.5  # Starting margin from the left
    padding = 0.6    # Space between different stacks
    
    for base_obj, stack in stacks.items():
        # Get the width of the widest item in this stack for centering
        max_w = max(get_width(world.objects[name]) for name in stack)
        center_x = current_x + (max_w / 2)
        stack_positions[base_obj] = center_x
        current_x += max_w + padding

    # 3. INITIALIZE FIGURE
    # Determine dynamic max_x. Minimum width of 5 to keep 'Holding' area visible.
    max_x_limit = max(6, current_x + 2) 
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.set_xlim(-0.5, max_x_limit)
    ax.set_ylim(-0.5, 5)
    ax.set_aspect('equal')
    ax.axis('off')

    # 4. DRAW STACKS
    for base_obj, stack in stacks.items():
        y_pos = 0
        center_x = stack_positions[base_obj]
        
        for obj_name in stack:
            obj = world.objects[obj_name]
            ##########
            patch = draw_object(obj, center_x, y_pos)
            ##########
            ax.add_patch(patch)
            # Center text in the shape
            ax.text(center_x, y_pos + get_height(obj)/2, obj_name, ha='center', va='center', 
                    fontsize=9, fontweight='bold')
            y_pos += get_height(obj)

    # 5. DRAW HELD OBJECT (Dynamic Position)
    if world.holding:
        obj = world.objects[world.holding]
        hold_x = max_x_limit - 1.5
        hold_y = 3.5
        
        patch = draw_object(obj, hold_x, hold_y)
        ax.add_patch(patch)
        ax.text(hold_x, hold_y - 0.4, f"HOLDING:\n{world.holding}", 
                ha='center', fontsize=10, fontweight='bold', color='red')

    # 6. TABLE LINE
    ax.axhline(y=0, color='brown', linewidth=4, alpha=0.7)
    ax.text(max_x_limit/2, -0.4, "TABLE", ha='center', fontsize=12, fontweight='bold', color='brown')

    # FINAL EXPORT
    plt.tight_layout()
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf)
    plt.close(fig)
    
    return img


In [77]:
# ----------------------------------------
# STEP 1) Processing
# ----------------------------------------

def process_command(command: str, history: List[List[str]]) -> Tuple[List[List[str]], Image.Image, str]:
    """Process a command and update the world."""
    global world_state
    
    if not command.strip():
        return history, visualize_world(world_state), world_state.describe()
    
    try:
        response = interpret_and_act(world_state, command)
        history.append([command, response])
    except (ValueError, RuntimeError) as e:
        response = f"❌ {str(e)}"
        history.append([command, response])
    
    return history, visualize_world(world_state), world_state.describe()

def reset_world():
    """Reset the world to initial state."""
    global world_state
    world_state = create_initial_world()
    return [], visualize_world(world_state), world_state.describe()

# Build Gradio interface
with gr.Blocks(title="SHRDLU Blocks World") as demo:
    gr.Markdown("""
    # 🧱 SHRDLU Blocks World Simulator
    
    A modern implementation of Terry Winograd's classic SHRDLU natural language understanding system.
    
    **Try commands like:**
    - "pick up the blue triangle"
    - "put the blue triangle on the red cube"
    - "put the green cube on the blue triangle"
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="Conversation History",
                height=400,
                show_label=True,
                # bubble_full_width=False
            )
            
            with gr.Row():
                command_input = gr.Textbox(
                    label="Enter Command",
                    placeholder="e.g., put the blue triangle on the red cube",
                    scale=4
                )
                submit_btn = gr.Button("Execute", variant="primary", scale=1)
            
            with gr.Row():
                reset_btn = gr.Button("🔄 Reset World", variant="secondary")
            
            world_text = gr.Textbox(
                label="World State (Text)",
                lines=6,
                interactive=False
            )
        
        with gr.Column(scale=1):
            world_viz = gr.Image(
                label="World Visualization",
                type="pil",
                height=500
            )
    
    gr.Markdown("""
    ### 📝 Notes:
    - Objects are identified by **color** and **shape**
    - The system will ask for clarification if the description is ambiguous
    - The planner automatically moves blocking objects out of the way
    - Dashed outline indicates an object being held
    """)
    
    # Event handlers
    submit_btn.click(
        fn=process_command,
        inputs=[command_input, chatbot],
        outputs=[chatbot, world_viz, world_text]
    ).then(
        lambda: "",
        outputs=[command_input]
    )
    
    command_input.submit(
        fn=process_command,
        inputs=[command_input, chatbot],
        outputs=[chatbot, world_viz, world_text]
    ).then(
        lambda: "",
        outputs=[command_input]
    )
    
    reset_btn.click(
        fn=reset_world,
        outputs=[chatbot, world_viz, world_text]
    )
    
    # Load initial state
    demo.load(
        fn=lambda: ([], visualize_world(world_state), world_state.describe()),
        outputs=[chatbot, world_viz, world_text]
    )

if __name__ == "__main__":
    demo.launch(share=False, server_name="0.0.0.0")


C:\Users\kyria\AppData\Local\Temp\ipykernel_10360\3079203733.py:42: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7860): only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7861): only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 7862): only one usage of each socket address (protocol/network address/port) is normally permitted
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.

* Running on local URL:  http://0.0.0.0:7869
* To create a public link, set `share=True` in `launch()`.


p1 c1
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
r1 None
s1 None
s2 None
l2 None
l3 l2
b3 l3
c3 None
c4 None
p1 c1
c1 None
b2 None
c2 None
l1 c2
